In [ ]:
using Plots
using Graphs
using GraphRecipes
using NetworkLayout  
using Random
using Statistics

In [ ]:
"""
Standardized Deffuant opinion update step.
Modifies `opinions` in-place. 
Returns `true` if an update occurred, `false` if bounded confidence failed.
"""
function deffuant_update!(opinions::Vector{Float64}, i::Int, j::Int, ϵ::Float64, μ::Float64, random_μ::Bool)
    if abs(opinions[i] - opinions[j]) < ϵ
        diff = opinions[j] - opinions[i]
        μ_eff = random_μ ? (rand() * 0.5 + 0.25) : μ
        
        opinions[i] += μ_eff * diff
        opinions[j] -= μ_eff * diff
        return true
    end
    return false
end

In [ ]:


"""
Simulates the Deffuant model for ANY static topology (Complete Graph, Lattices, etc.)
"""
function simulate_deffuant_static(g::AbstractGraph, MCS::Int, ϵ::Float64, μ::Float64; random_μ::Bool=true)
    N = nv(g)
    opinions = rand(N) .* 2.0 .- 1.0
    history = zeros(Float64, MCS + 1, N)
    history[1, :] .= opinions

    for t in 1:MCS
        for _ in 1:N
            i = rand(1:N)
            nbrs = neighbors(g, i)
            if !isempty(nbrs)
                j = rand(nbrs)
                # Call modular step function
                deffuant_update!(opinions, i, j, ϵ, μ, random_μ) 
            end
        end
        history[t+1, :] .= opinions
    end

    return history
end

"""
Simulates the Adaptive Deffuant model where frustrated links can be homophilically rewired.
"""
function simulate_deffuant_adaptive(g_initial::AbstractGraph, MCS::Int, ϵ::Float64, μ::Float64, w::Float64; save_interval::Int=10, random_μ::Bool=false)
    N = nv(g_initial)
    g = copy(g_initial) # Copy so we don't mutate the original input
    opinions = rand(N) .* 2.0 .- 1.0
    
    history = zeros(Float64, MCS + 1, N)
    history[1, :] .= opinions
    
    graph_history = typeof(g)[]
    push!(graph_history, copy(g))

    for t in 1:MCS
        for _ in 1:N
            i = rand(1:N)
            nbrs = neighbors(g, i)

            if !isempty(nbrs)
                j = rand(nbrs)
                # Call modular step function
                updated = deffuant_update!(opinions, i, j, ϵ, μ, random_μ)
                
                # Structural update if opinions are frustrated
                if !updated && rand() < w
                    for attempt in 1:50 # limit attempts
                        k = rand(1:N)
                        if k != i && k != j && !has_edge(g, i, k) && abs(opinions[i] - opinions[k]) <= ϵ
                            rem_edge!(g, i, j)
                            add_edge!(g, i, k)
                            break
                        end
                    end
                end
            end
        end

        history[t+1, :] .= opinions
        if t % save_interval == 0
            push!(graph_history, copy(g))
        end
    end

    return history, graph_history
end

In [ ]:

"""
    make_generalized_lattice(Lx::Int, Ly::Int, neighbor_rule::Function; 
                             coord_rule::Function=(x,y)->(x,y), periodic::Bool=true)

Returns: (g::SimpleGraph, locs_x::Vector{Float64}, locs_y::Vector{Float64})
"""
function make_generalized_lattice(Lx::Int, Ly::Int, neighbor_rule::Function; 
                                  coord_rule::Function=(x,y)->(Float64(x), Float64(y)), 
                                  periodic::Bool=true)
    g = SimpleGraph(Lx * Ly)
    locs_x = zeros(Float64, Lx * Ly)
    locs_y = zeros(Float64, Lx * Ly)
    
    node_idx(x, y) = (x - 1) * Ly + y
    
    for x in 1:Lx, y in 1:Ly
        u = node_idx(x, y)
        
        # Save actual spatial coordinates
        cx, cy = coord_rule(x, y)
        locs_x[u] = cx
        locs_y[u] = cy
        
        offsets = neighbor_rule(x, y)
        for (dx, dy) in offsets
            nx, ny = x + dx, y + dy
            
            if periodic
                nx = mod1(nx, Lx)
                ny = mod1(ny, Ly)
            else
                if nx < 1 || nx > Lx || ny < 1 || ny > Ly
                    continue
                end
            end
            
            v = node_idx(nx, ny)
            if u != v
                add_edge!(g, u, v)
            end
        end
    end
    return g, locs_x, locs_y
end

In [ ]:


# A) Plot standard opinion trajectories
function plot_opinion_evolution(history; title="Opinion Evolution", color_grad=:RdBu)
    MCS = size(history, 1) - 1
    N = size(history, 2)
    final_opinions = history[end, :]
    
    p = plot(title=title, xlabel="Time (MCS)", ylabel="Opinion", ylims=(-1.05, 1.05), legend=false, grid=true)
    for i in 1:N
        color_val = (final_opinions[i] + 1.0) / 2.0
        plot!(p, 0:MCS, history[:, i], linecolor=cgrad(color_grad)[color_val], linealpha=0.3, linewidth=1.2)
    end
    return p
end

function plot_network_snapshot(g, opinions; x_coords=nothing, y_coords=nothing, 
                               title="Network Snapshot", color_grad=:RdBu,
                               node_size=nothing, scale_by_degree=nothing)
    
    # Auto-detect if it's a regular grid based on coordinates
    is_regular_grid = !isnothing(x_coords) && !isnothing(y_coords)
    
    # Default behavior: scale BA networks by degree, keep regular grids uniform
    do_scale = isnothing(scale_by_degree) ? !is_regular_grid : scale_by_degree
    
    # Set sensible default base sizes if not provided
    if isnothing(node_size)
        base_size = is_regular_grid ? 5.0 : 4.0 
    else
        base_size = Float64(node_size)
    end
    
    # Calculate final marker sizes
    if do_scale
        node_degrees = Float64.(degree(g))
        markersizes = base_size .+ 16.0 .* (node_degrees ./ (maximum(node_degrees) + 1e-6))
    else
        # Uniform size for all nodes (perfect for lattices)
        markersizes = fill(base_size, nv(g))
    end

    # Plot the underlying edges
    if is_regular_grid
        p_net = graphplot(g, x=x_coords, y=y_coords, size=(500, 500), 
                          markersize=0.01, markeralpha=0.0, linecolor=:gray, linealpha=0.5, 
                          title=title, colorbar=false, axis=false, grid=false)
    else
        p_net = graphplot(g, size=(500, 500), markersize=0.01, markeralpha=0.0,
                          linecolor=:gray, linealpha=0.1, method=:sfdp,
                          title=title, colorbar=false, axis=false, grid=false)
    end
    
    # Overlay our custom sized and colored nodes
    node_series = p_net.series_list[end]
    scatter!(p_net, node_series[:x], node_series[:y],
             markersize=markersizes, marker_z=opinions, markercolor=color_grad,
             clims=(-1.0, 1.0), markershape=:circle, markerstrokewidth=0.5,
             markerstrokecolor=:black, label="")
             
    return p_net
end

# C) Combine into your big Adaptive Plotter easily
function plot_adaptive_snapshots(history, graph_history, save_interval, w)
    p_time = plot_opinion_evolution(history, title="Opinion Evolution (w = $w)")
    
    num_snaps = length(graph_history)
    snapshots_to_show = [1, max(2, num_snaps ÷ 2), num_snaps]
    titles = ["Initial (t=0)", "Intermediate", "Final State"]
    
    plots = []
    for (idx, snap_idx) in enumerate(snapshots_to_show)
        g_snap = graph_history[snap_idx]
        ops_snap = history[(snap_idx-1)*save_interval+1, :]
        push!(plots, plot_network_snapshot(g_snap, ops_snap, title=titles[idx]))
    end
    
    final_layout = plot(plots..., layout=(1, length(plots)), size=(1500, 500))
    display(p_time)
    display(final_layout)
end

In [ ]:
# 1. Standard 1D Complete Mixing (like your 1st original function)
g_complete = complete_graph(1000) 
history_1d = simulate_deffuant_static(g_complete, 1000, 0.2, 0.5, random_μ=true)

# 3. Barabási-Albert for the Adaptive logic
g_ba = barabasi_albert(500, 6)
history_ba, g_hist = simulate_deffuant_adaptive(g_ba, 100, 0.4 , 0.5 , 0.4, save_interval=1 ,random_μ=true )

In [ ]:
plot_adaptive_snapshots(history_ba, g_hist, 1, 0.5)

In [ ]:
von_neumann_rule(x, y) = [(1, 0), (-1, 0), (0, 1), (0, -1)]
function hexagonal_rule(x, y)
    # Everyone connects up and down
    neighbors = [(0, 1), (0, -1)]
    
    # Alternate left/right connections to form hexagons
    if iseven(x + y)
        push!(neighbors, (1, 0))  # connect right
    else
        push!(neighbors, (-1, 0)) # connect left
    end
    
    return neighbors
end

In [ ]:
# Standard Cartesian coordinates for Square, Moore, Triangular
square_coords(x, y) = (Float64(x), Float64(y))

# Slanted coordinates so the "brick wall" topology looks like perfect hexagons
function hex_coords(x, y)
    cx = x * sqrt(3)
    # Stagger every other column
    if iseven(y)
        cx += sqrt(3)/2
    end
    cy = y * 1.5
    return (cx, cy)
end

In [ ]:
# 1. Generate Lattices WITH coordinates
L = 20
g_square, sx, sy = make_generalized_lattice(L, L, von_neumann_rule, coord_rule=square_coords)
g_hex, hx, hy    = make_generalized_lattice(L, L, hexagonal_rule, coord_rule=hex_coords)

# 2. Simulate
hist_square = simulate_deffuant_static(g_square, 1000, 0.2, 0.5)
hist_hex    = simulate_deffuant_static(g_hex, 1000, 0.2, 0.5)

# 3. Plot WITH exact coordinates
final_ops_square = hist_square[end, :]
final_ops_hex    = hist_hex[end, :]

p_grid= plot_network_snapshot(g_square, final_ops_square, x_coords=sx, y_coords=sy, node_size=10)
p_honey = plot_network_snapshot(g_hex, final_ops_hex, x_coords=hx, y_coords=hy, title="Hexagonal Lattice")

display(p_grid)
display(p_honey)

In [ ]:
p_dense = plot_network_snapshot(g_square, final_ops_square, x_coords=sx, y_coords=sy, node_size=2.5)

In [ ]:


"""
    largest_cluster_size(opinions::AbstractVector{Float64}, threshold::Float64=1e-3)

Calculates the size of the largest opinion cluster.
Agents are considered part of the same cluster if their sorted opinions 
differ by less than the specified threshold.
"""
function largest_cluster_size(opinions::AbstractVector{Float64}, threshold::Float64=1e-3)
    sorted_ops = sort(opinions)
    max_cluster = 1
    current_cluster = 1
    
    for i in 2:length(sorted_ops)
        if sorted_ops[i] - sorted_ops[i-1] < threshold
            current_cluster += 1
        else
            if current_cluster > max_cluster
                max_cluster = current_cluster
            end
            current_cluster = 1  # Reset for the next cluster
        end
    end
    
    # Catch the final cluster if it happens to be the largest
    if current_cluster > max_cluster
        max_cluster = current_cluster
    end
    
    return max_cluster
end

function calculate_ensemble_observables(final_Os::AbstractVector{Float64}, N::Int)
    O_mean = mean(final_Os)
    O2_mean = mean(final_Os.^2)
    O4_mean = mean(final_Os.^4)
    
    # 1. Order Parameter (Ensemble Average)
    O = O_mean
    
    # 2. Susceptibility (Ensemble Variance scaled by N)
    χ = N * (O2_mean - O_mean^2)
    
    # 3. Binder Cumulant (Ensemble 4th order moment)
    U4 = 1.0 - (O4_mean / (3.0 * O2_mean^2 + 1e-12))
    
    return O, χ, U4
end

"""
    scan_thermodynamics_ensemble(N::Int, ϵ_range, MCS::Int, μ::Float64, num_trials::Int; kwargs...)

Sweeps over tolerance `ϵ` and performs `num_trials` independent simulations per ϵ.
Calculates thermodynamic observables across the ensemble.
"""
function scan_thermodynamics_ensemble(g, N, ϵ_range, MCS, μ, num_trials; 
                                      cluster_threshold = 1e-3, w=0.0, adaptive=false, random_μ=true)
    
    # Pre-allocate arrays for the results
    O_avg = zeros(length(ϵ_range))
    χ_avg = zeros(length(ϵ_range))
    U4_avg = zeros(length(ϵ_range))
    sum_abs_avg = zeros(length(ϵ_range))
    num_clusters_avg = zeros(length(ϵ_range))

    for (i, ϵ) in enumerate(ϵ_range)
        O_list = zeros(num_trials)
        sum_abs_list = zeros(num_trials)
        num_clusters_list = zeros(num_trials)

        for trial in 1:num_trials
            # Run the simulation
            if adaptive
                history, _ = simulate_deffuant_adaptive(g, MCS, ϵ, μ, w; random_μ=random_μ, save_interval=MCS)
            else
                history = simulate_deffuant_static(g, MCS, ϵ, μ; random_μ=random_μ)
            end
            
            final_opinions = history[end, :]
            
            # === EXTRACT CLUSTER SIZES ===
            sorted_ops = sort(final_opinions)
            cluster_sizes = Int[]
            current_cluster_size = 1
            
            for j in 2:length(sorted_ops)
                # If gap is smaller than threshold, they belong to the same cluster
                if sorted_ops[j] - sorted_ops[j-1] < cluster_threshold
                    current_cluster_size += 1
                else
                    # Gap found! Save the size of the cluster we just finished tracking
                    push!(cluster_sizes, current_cluster_size)
                    current_cluster_size = 1 # Reset for the next cluster
                end
            end
            # Don't forget to push the very last cluster
            push!(cluster_sizes, current_cluster_size)
            
            # 1. Order Parameter (Largest cluster fraction)
            max_cluster = maximum(cluster_sizes)
            O_list[trial] = max_cluster / N
            
            # 2. Sum of Absolute Opinions
            sum_abs_list[trial] = sum(abs.(final_opinions))
            
            # 3. WEIGHTED Number of Clusters (Effective Number of Clusters)
            p = cluster_sizes ./ N           # Calculate the fraction of each cluster
            N_eff = 1.0 / sum(p.^2)          # Calculate the effective cluster count
            num_clusters_list[trial] = N_eff
        end
        
        # --- Thermodynamics Averages ---
        O_mean = mean(O_list)
        O2_mean = mean(O_list.^2)
        O4_mean = mean(O_list.^4)
        
        O_avg[i] = O_mean
        χ_avg[i] = N * (O2_mean - O_mean^2)
        U4_avg[i] = O2_mean == 0 ? 0.0 : 1.0 - (O4_mean / (3.0 * O2_mean^2))
        
        sum_abs_avg[i] = mean(sum_abs_list)
        num_clusters_avg[i] = mean(num_clusters_list)
    end
    
    return O_avg, χ_avg, U4_avg, sum_abs_avg, num_clusters_avg
end

In [ ]:

# ==========================================
# 1. Define Simulation Parameters
# ==========================================
N =400                  # Number of agents (population size)
MCS = 1000               # Monte Carlo Steps (time steps)
μ = 0.5       
random_μ=true           # Convergence parameter
ϵ_range = 0.3:0.01:0.6   # Sweep tolerance from 0.1 to 0.6
w=0.5

# Create a fully connected society (Mean-Field approximation)
# If you prefer a network, you could use erdos_renyi_graph(N, 0.1) etc.
#g = barabasi_albert(500, 6)
L = 20
# g_square, _, __ = make_generalized_lattice(L, L, von_neumann_rule, coord_rule=square_coords)
# g= g_square
g = barabasi_albert(400, 4)
g=complete_graph(400)

num_trials = 50          # Number of independent realizations per ϵ

println("Running $num_trials trials per ϵ (Total simulations: $(length(ϵ_range) * num_trials))...")

# Make sure your function returns the two new arrays!
O_vals, χ_vals, U4_vals, sum_abs_vals, num_clusters_vals = scan_thermodynamics_ensemble(
    g, N, ϵ_range, MCS, μ, num_trials; 
    cluster_threshold = 1e-2, 
    w=w,
    adaptive = false,
    random_μ=true
)

println("Scan complete! Generating plots...")

# ==========================================
# Create the Plots
# ==========================================
println("Scan complete! Generating plots...")

# ==========================================
# Create the Plots
# ==========================================
p1 = plot(ϵ_range, O_vals, marker=:circle, color=:blue, ylabel="Order Parameter (O)", title="Ensemble Phase Transition (N=$N)", legend=false, linewidth=2)

p2 = plot(ϵ_range, χ_vals, marker=:square, color=:red, ylabel="Susceptibility (χ)", legend=false, linewidth=2)

p3 = plot(ϵ_range, U4_vals, marker=:diamond, color=:green, ylabel="Binder Cumulant (U₄)", legend=false, linewidth=2)

p4 = plot(ϵ_range, sum_abs_vals, marker=:hexagon, color=:orange, ylabel="Sum of |Opinions|", legend=false, linewidth=2)

# ---> NEW: Create labels from the y-values (rounded to 1 decimal place, placed slightly above the point)
cluster_labels = text.(round.(num_clusters_vals, digits=1), 8, :bottom)

# ---> NEW: Added series_annotations to draw the labels on the plot
p5 = plot(ϵ_range, num_clusters_vals, 
          marker=:star, color=:purple, 
          xlabel="Tolerance (ϵ)", ylabel="Number of Clusters", 
          legend=false, linewidth=2,
          series_annotations=cluster_labels)

# Use a taller layout to give the text space to breathe
final_plot = plot(p1, p2, p3, p4, p5, layout=(5, 1), size=(800, 1100))

display(final_plot)